# RailPulse SQL Analysis

Use this notebook to develop and test your own SQL queries against `railpulse.db`.

The notebook provides instructions but does not contain answers. Save each completed query in its corresponding file under `sql/analysis/`.

## Setup: connect to SQLite

Run this cell once before starting the five analysis sessions.

In [12]:
import sqlite3
from pathlib import Path


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE_PATH = PROJECT_ROOT / "database" / "railpulse.db"

print(f"Database: {DATABASE_PATH}")
print(f"Exists: {DATABASE_PATH.is_file()}")

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DATABASE_PATH}")

connection = sqlite3.connect(DATABASE_PATH)
cursor = connection.cursor()

Database: c:\BeCode_data\railpulse_sql_analysis\database\railpulse.db
Exists: True


## Query helper

Pass your SQL string to `run_query()` to display its column names and rows. This helper executes your SQL but does not contain analytical logic.

In [13]:
def run_query(query):
    """Execute a SELECT query and display its result."""

    if not query.strip():
        print("Write a SQL query before running this cell.")
        return

    result_cursor = connection.execute(query)

    column_names = [
        column[0]
        for column in result_cursor.description
    ]

    rows = result_cursor.fetchall()

    print(column_names)

    for row in rows:
        print(row)

    print(f"Rows returned: {len(rows)}")

# Session 1 — The Peak Hour Problem

**Question:** What hour of the day experiences the highest volume of scheduled train departures across the entire network?

### Relevant data

- Table: `stop_times`
- Column: `departure_time`

### Instructions

1. Inspect several `departure_time` values with `LIMIT`.
2. Extract the hour portion of each time.
3. Remember that GTFS times can use hours greater than 23 for service after midnight.
4. Decide how to represent those values as an hour of the day.
5. Count scheduled departure records for each hour.
6. Group the records by the calculated hour.
7. Sort from the largest count to the smallest.
8. Return only the busiest hour.

### SQL concepts to practise

`SELECT`, `SUBSTR`, `CAST`, arithmetic, `COUNT`, `GROUP BY`, `ORDER BY`, `LIMIT`

Save the final query as `sql/analysis/01_peak_hour.sql`.

In [47]:
question_1_query = """
-- Write and test your Question 1 SQL here.

WITH leaving_time AS(
SELECT CAST(SUBSTR(departure_time, 1,2) AS INTEGER)%24 AS onlyhour
FROM stop_times 
)

SELECT onlyhour, COUNT(onlyhour) AS departure_count
FROM leaving_time
GROUP BY onlyhour
ORDER BY departure_count DESC
LIMIT 3;


"""

#run_query(question_1_query)

# Remove the comment marker below when your query is ready.
run_query(question_1_query)

QUERY_PATH = (
    PROJECT_ROOT
    / "sql"
    / "analysis"
    / "01_peak_hour.sql"
)

QUERY_PATH.write_text(
    question_1_query.strip() + "\n",
    encoding="utf-8",
)

print(f"Saved: {QUERY_PATH}")

['onlyhour', 'departure_count']
(10, 139071)
Rows returned: 1
Saved: c:\BeCode_data\railpulse_sql_analysis\sql\analysis\01_peak_hour.sql


# Session 2 — Platform Bottlenecks

**Question:** Identify the top 3 busiest platforms in Brussels-Central.

### Relevant data

- Tables: `stops`, `stop_times`
- Stop columns to investigate: `stop_id`, `stop_name`, `parent_station`, `location_type`, `platform_code`
- Connection: `stop_times.stop_id` references `stops.stop_id`

### Instructions

1. Search `stops` to find how Brussels-Central is represented in the data.
2. Inspect its parent-station and platform records.
3. Determine which column identifies an individual platform.
4. Join platform stops to their scheduled stop-time records.
5. Count scheduled activity for each platform.
6. Group by stable platform identifiers as well as the display label.
7. Sort from busiest to least busy.
8. Return the top three platforms.

### SQL concepts to practise

`WHERE`, `LIKE`, aliases, `JOIN`, `ON`, `COUNT`, `GROUP BY`, `ORDER BY`, `LIMIT`

Save the final query as `sql/analysis/02_platform_bottlenecks.sql`.

In [46]:
question_2_query = """
SELECT COUNT(*) AS scheduled_visits, platform_code
FROM stop_times
JOIN stops on stop_times.stop_id = stops.stop_id 
WHERE stop_name ='Bruxelles-Central' AND platform_code IS NOT NULL 
GROUP BY platform_code
ORDER BY scheduled_visits DESC
LIMIT 3;

"""

# Remove the comment marker below when your query is ready.
run_query(question_2_query)

QUERY_PATH = (
    PROJECT_ROOT
    / "sql"
    / "analysis"
    / "02_platform_bottlenecks.sql"
)

QUERY_PATH.write_text(
    question_2_query.strip() + "\n",
    encoding="utf-8",
)

print(f"Saved: {QUERY_PATH}")

['scheduled_visits', 'platform_code']
(11982, '3')
(10515, '4')
(7473, '2')
Rows returned: 3
Saved: c:\BeCode_data\railpulse_sql_analysis\sql\analysis\02_platform_bottlenecks.sql


# Session 3 — Busiest Morning Destinations

**Question:** Find the top 3 most frequent terminal destinations (`trip_headsign`) for all morning trips that depart before 12:00:00 PM.

### Relevant data

- Tables: `trips`, `stop_times`
- Trip columns: `trip_id`, `trip_headsign`
- Time columns: `trip_id`, `stop_sequence`, `departure_time`

### Instructions

1. Decide what counts as the departure time of a trip rather than every intermediate stop departure.
2. Use `stop_sequence` or an aggregate to identify the first scheduled departure for each trip.
3. Filter those trip departures to the morning period before noon.
4. Join the morning trips to `trips` to obtain `trip_headsign`.
5. Exclude missing destination labels if necessary.
6. Count trips for each destination.
7. Sort by frequency and return the top three.

### SQL concepts to practise

`JOIN`, subquery or CTE, `MIN`, `WHERE`, `COUNT`, `GROUP BY`, `ORDER BY`, `LIMIT`

Save the final query as `sql/analysis/03_morning_destinations.sql`.

In [45]:
question_3_query = """
-- Write and test your Question 3 SQL here.
WITH first_stop AS 
(
SELECT 
    trip_id,
    MIN(stop_sequence) AS first_stop_sequence
FROM stop_times
GROUP BY trip_id
),

trip_departure AS (
    SELECT
        stop_times.trip_id,
        trips.trip_headsign,
        CAST(
            SUBSTR(stop_times.departure_time, 1, 2)
            AS INTEGER
        ) % 24 AS departure_hour
    FROM first_stop
    JOIN stop_times
        ON stop_times.trip_id = first_stop.trip_id 
        AND stop_times.stop_sequence = first_stop.first_stop_sequence 
    JOIN trips 
        ON stop_times.trip_id = trips.trip_id
)

SELECT trip_headsign, COUNT(*) AS counted_trips
FROM trip_departure
WHERE departure_hour < 12  
GROUP BY trip_headsign
ORDER BY counted_trips DESC 
LIMIT 3;


"""

# Remove the comment marker below when your query is ready.
run_query(question_3_query)

QUERY_PATH = (
    PROJECT_ROOT
    / "sql"
    / "analysis"
    / "03_platform_bottlenecks.sql"
)

QUERY_PATH.write_text(
    question_3_query.strip() + "\n",
    encoding="utf-8",
)

print(f"Saved: {QUERY_PATH}")



['trip_headsign', 'counted_trips']
('Anvers-Central', 3939)
('Bruxelles-Midi', 3155)
('Louvain', 2507)
Rows returned: 3
Saved: c:\BeCode_data\railpulse_sql_analysis\sql\analysis\03_platform_bottlenecks.sql


# Session 4 — Service Frequency

**Question:** Classify each active service ID into a weekly frequency category using `CASE WHEN`. Services operating 5 or more days are `High Frequency`; 2–4 days are `Medium Frequency`; and 1 day or completely irregular services are `Low Frequency/Special`. Show the percentage of services in each category.

### Relevant data

- Table: `services`
- Columns: `service_id`, `monday`, `tuesday`, `wednesday`, `thursday`, `friday`, `saturday`, `sunday`

### Instructions

1. Calculate the number of regular operating days for every service.
2. Use `CASE WHEN` to assign the required category.
3. Keep this row-level classification as an intermediate query.
4. Group the classified services by category.
5. Count the services in each category.
6. Calculate each category's percentage of the total service count.
7. Prevent integer division when calculating the percentage.
8. Consider how a service with zero regular weekdays should be classified.

### SQL concepts to practise

Arithmetic, `CASE WHEN`, CTE or subquery, `COUNT`, `GROUP BY`, percentage calculation, `ROUND`

Save the final query as `sql/analysis/04_service_frequency.sql`.

In [53]:
question_4_query = """
-- Write and test your Question 4 SQL here.
WITH active_service AS (
SELECT DISTINCT service_id 
FROM trips 
),

service_datefinder AS (
    SELECT service_exceptions.service_id, exception_date
    FROM service_exceptions 
    JOIN active_service
        ON  active_service.service_id = service_exceptions.service_id
    WHERE exception_type =1 
    
),

weekly_activity AS (
SELECT 
    service_id, 
    STRFTIME('%Y-%W', exception_date) AS active_week,
    COUNT(DISTINCT exception_date) AS active_days_in_week
FROM service_datefinder
GROUP BY 
    service_id,
    active_week
),

average_active AS (
SELECT 
    service_id,
    AVG(active_days_in_week) AS average_days_per_week
FROM weekly_activity 
GROUP BY service_id 

)


SELECT
    CASE
        WHEN average_days_per_week >= 5
            THEN 'High frequency'
        WHEN average_days_per_week < 2
            THEN 'Low frequency/special'
        ELSE 'Medium frequency'
    END AS frequency,
    COUNT(*) AS number_of_services,
    ROUND( 100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_services

FROM average_active
GROUP BY frequency
ORDER BY percentage_of_services DESC;

"""

QUERY_PATH = (
    PROJECT_ROOT
    / "sql"
    / "analysis"
    / "04_serivce_frequency.sql"
)

QUERY_PATH.write_text(
    question_4_query.strip() + "\n",
    encoding="utf-8",
)

print(f"Saved: {QUERY_PATH}")
# Remove the comment marker below when your query is ready.
run_query(question_4_query)

Saved: c:\BeCode_data\railpulse_sql_analysis\sql\analysis\04_serivce_frequency.sql
['frequency', 'number_of_services', 'percentage_of_services']
('Medium frequency', 10835, 62.67)
('Low frequency/special', 6046, 34.97)
('High frequency', 407, 2.35)
Rows returned: 3


# Session 5 — The Accessibility Audit

**Question:** Calculate the exact ratio and percentage of scheduled trips per route that explicitly guarantee wheelchair accessibility or bicycle storage. Which routes score the lowest in passenger amenity availability?

### Relevant data

- Tables: `routes`, `trips`
- Route columns: `route_id`, `route_short_name`, `route_long_name`
- Feature columns: `wheelchair_accessible`, `bikes_allowed`

### Instructions

1. Inspect the distinct values and null counts in both feature columns.
2. Check the GTFS meaning of each code before deciding what counts as an explicit guarantee.
3. Join trips to their routes.
4. Count all scheduled trips per route for the denominator.
5. Use conditional aggregation to count qualifying trips for the numerator.
6. Return the exact numerator and denominator as the ratio components.
7. Calculate a percentage without integer division.
8. Sort ascending to identify the lowest-scoring routes.
9. Decide how ties and routes with no explicit feature information should be presented.

### SQL concepts to practise

`JOIN`, `CASE WHEN`, conditional aggregation, `COUNT`, `SUM`, `GROUP BY`, percentage calculation, `ROUND`, `ORDER BY`

Save the final query as `sql/analysis/05_accessibility_audit.sql`.

In [69]:

bikes_query = """
SELECT
    bikes_allowed,
    COUNT(*) AS number_of_trips
FROM trips
GROUP BY bikes_allowed;
"""

wheelchair_query = """
SELECT
    wheelchair_accessible,
    COUNT(*) AS number_of_trips
FROM trips
GROUP BY wheelchair_accessible;
"""

trip_query = """
SELECT
    CASE
        WHEN trip_id LIKE '%BBUS%'
            THEN 'Possible replacement bus'
        ELSE 'Other trip'
    END AS trip_group,
    bikes_allowed,
    COUNT(*) AS number_of_trips
FROM trips
GROUP BY
    trip_group,
    bikes_allowed
ORDER BY
    trip_group,
    bikes_allowed;

"""

#run_query(trip_query)
#run_query(wheelchair_query)

#wheelchair_accessible = NULL for all 134,809 trips

question_5_query = """
-- Write and test your Question 5 SQL here.
WITH trip_features AS (
    SELECT t.route_id, 
        r.route_short_name, 
        r.route_long_name, 
        t.bikes_allowed,
        t.wheelchair_accessible,
        CASE 
            WHEN r.route_short_name = 'BUS'
                THEN 'Bus service'
            ELSE 'Train service'
        END AS service_type
    FROM trips AS t 
    JOIN routes AS r
        ON t.route_id = r.route_id 
),

route_amenities AS (
    SELECT
        route_id,
        route_short_name,
        route_long_name,
        service_type,
        COUNT(*) AS total_trip,
        SUM(
            CASE
                WHEN bikes_allowed = 1 THEN 1
                ELSE 0
            END
        ) AS bike_guaranteed_trips,

        SUM(
            CASE
                WHEN bikes_allowed IS NULL THEN 1
                ELSE 0
            END 
        ) AS bike_unknown_trips,
                SUM(
            CASE
                WHEN wheelchair_accessible = 1 THEN 1
                ELSE 0
            END
        ) AS wheelchair_guaranteed_trips,

        SUM(
            CASE
                WHEN wheelchair_accessible IS NULL THEN 1
                ELSE 0
            END
        ) AS wheelchair_unknown_trips,
        SUM(
            CASE
                WHEN wheelchair_accessible = 1
                OR bikes_allowed = 1
            THEN 1
            ELSE 0
        END
        ) AS amenity_guaranteed_trips

    FROM trip_features
    GROUP BY
        route_id,
        route_short_name,
        route_long_name,
        service_type

)

SELECT
    route_id,
    route_short_name,
    route_long_name,
    service_type,
    total_trip,
    amenity_guaranteed_trips,
    amenity_guaranteed_trips || '/' || total_trip AS amenity_guarantee_ratio,
    ROUND(
        100.0 * amenity_guaranteed_trips / total_trip,
        2
    ) AS amenity_guarantee_percentage,
    bike_unknown_trips,
    wheelchair_unknown_trips
FROM route_amenities
ORDER BY 
    amenity_guarantee_percentage ASC,
    TOTAL_trip DESC;
"""
# Remove the comment marker below when your query is ready.
run_query(question_5_query)

['route_id', 'route_short_name', 'route_long_name', 'service_type', 'total_trip', 'amenity_guaranteed_trips', 'amenity_guarantee_ratio', 'amenity_guarantee_percentage', 'bike_unknown_trips', 'wheelchair_unknown_trips']
('gr:nmbssncb:68', 'BUS', 'Luxembourg (LU) -- Arlon', 'Bus service', 663, 0, '0/663', 0.0, 663, 663)
('gr:nmbssncb:1393', 'BUS', 'Bruxelles-Midi -- Nivelles', 'Bus service', 331, 0, '0/331', 0.0, 331, 331)
('gr:nmbssncb:1537', 'BUS', 'Charleroi-Central -- Fleurus', 'Bus service', 229, 0, '0/229', 0.0, 229, 229)
('gr:nmbssncb:1377', 'BUS', 'Hasselt -- Kiewit', 'Bus service', 203, 0, '0/203', 0.0, 203, 203)
('gr:nmbssncb:1538', 'BUS', 'Ottignies -- Fleurus', 'Bus service', 174, 0, '0/174', 0.0, 174, 174)
('gr:nmbssncb:1430', 'BUS', 'Zottegem -- Denderleeuw', 'Bus service', 173, 0, '0/173', 0.0, 173, 173)
('gr:nmbssncb:1548', 'BUS', 'Liège-Guillemins -- Flémalle-Haute', 'Bus service', 167, 0, '0/167', 0.0, 167, 167)
('gr:nmbssncb:15', 'BUS', 'Breda (NL) -- Noorderkempen (Br

## Finish the analysis session

Run the cell below when you are finished working in the notebook.

In [ ]:
connection.close()
print("Database connection closed.")